In [1]:
import pandas as pd
import numpy as np
import torch

In [2]:
class Feedforward(torch.nn.Module):
        def __init__(self, input_size, hidden_size, dropout):
            super(Feedforward, self).__init__()
            self.input_size = input_size
            self.hidden_size  = hidden_size
            self.fc_in = torch.nn.Linear(self.input_size, self.hidden_size)
            self.fc1 = torch.nn.Linear(self.hidden_size, self.hidden_size)
            self.fc2 = torch.nn.Linear(self.hidden_size, self.hidden_size)
            self.relu = torch.nn.ReLU()
            #self.relu = torch.nn.LeakyReLU(0.1)
            self.fc_out = torch.nn.Linear(self.hidden_size, 1)
            self.sigmoid = torch.nn.Sigmoid()
            self.dropout = torch.nn.Dropout(dropout)
            
        def forward(self, x):
            x = self.fc_in(x)
            x = self.dropout(x)
            x = self.relu(x)
            x = self.fc1(x)
            x = self.dropout(x)
            x = self.relu(x)
            x = self.fc2(x)
            x = self.dropout(x)
            x = self.relu(x)
            output = self.fc_out(x)
            output = self.sigmoid(output)
            return output

In [3]:
data = pd.read_csv('/Users/nkerstingadxnet.com/Documents/Higgs/orig/atlas-higgs-challenge-2014-v2.csv')
data['Binary_Label'] = data['Label'].map({'s':1,'b':0})

In [4]:
nonphi_columns = [c for c in data.columns if "phi" in c]
for col in nonphi_columns:
    data.drop(col, axis=1, inplace=True)
data.head()

,EventId,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,...,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_all_pt,Weight,Label,KaggleSet,KaggleWeight,Binary_Label
0,100000,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,...,67.435,2.150,46.062,1.24,113.497,0.000814,s,t,0.002653,1
1,100001,160.937,68.768,103.235,48.146,-999.00,-999.000,-999.000,3.473,2.078,...,46.226,0.725,-999.000,-999.00,46.226,0.681042,b,t,2.233584,0
2,100002,-999.000,162.172,125.953,35.635,-999.00,-999.000,-999.000,3.148,9.336,...,44.251,2.053,-999.000,-999.00,44.251,0.715742,b,t,2.347389,0
3,100003,143.905,81.417,80.943,0.414,-999.00,-999.000,-999.000,3.310,0.414,...,-999.000,-999.000,-999.000,-999.00,-0.000,1.660654,b,t,5.446378,0
4,100004,175.864,16.915,134.805,16.405,-999.00,-999.000,-999.000,3.891,16.405,...,-999.000,-999.000,-999.000,-999.00,0.000,1.904263,b,t,6.245333,0


In [5]:
# now replace all the '-999' values with NULL so we can properly handle them
#nulled_data = data.replace(-999.0, np.nan)
nulled_data = data.replace(-999.0, 0)
train_data = nulled_data.loc[nulled_data['KaggleSet'] == 't'].copy(deep=True)
public_test_data = nulled_data.loc[nulled_data['KaggleSet'] == 'b'].copy(deep=True)
private_test_data = nulled_data.loc[nulled_data['KaggleSet'] == 'v'].copy(deep=True)

In [6]:
orig_train_data = train_data.copy(deep=True)
orig_public_test_data = public_test_data.copy(deep=True)
orig_private_test_data = private_test_data.copy(deep=True)


# now replace the null values with column averages
train_data.drop('EventId', axis=1, inplace=True)
train_data.drop('Label', axis=1, inplace=True)
train_data.drop('Binary_Label', axis=1, inplace=True)
train_data.drop('Weight', axis=1, inplace=True)
train_data.drop('KaggleWeight', axis=1, inplace=True)
train_data.drop('KaggleSet', axis=1, inplace=True)
for col in train_data:
    train_data[col].fillna(train_data[col].mean(), inplace=True)
#train_data.isnull().sum()


public_test_data.drop('EventId', axis=1, inplace=True)
public_test_data.drop('Label', axis=1, inplace=True)
public_test_data.drop('Binary_Label', axis=1, inplace=True)
public_test_data.drop('Weight', axis=1, inplace=True)
public_test_data.drop('KaggleWeight', axis=1, inplace=True)
public_test_data.drop('KaggleSet', axis=1, inplace=True)
for col in public_test_data:
    public_test_data[col].fillna(public_test_data[col].mean(), inplace=True)
#public_test_data.isnull().sum()


private_test_data.drop('EventId', axis=1, inplace=True)
private_test_data.drop('Label', axis=1, inplace=True)
private_test_data.drop('Binary_Label', axis=1, inplace=True)
private_test_data.drop('Weight', axis=1, inplace=True)
private_test_data.drop('KaggleWeight', axis=1, inplace=True)
private_test_data.drop('KaggleSet', axis=1, inplace=True)
for col in private_test_data:
    private_test_data[col].fillna(private_test_data[col].mean(), inplace=True)
#private_test_data.isnull().sum()

In [7]:
# now let's normalize
normed_train_data = (train_data - train_data.min())/(train_data.max() - train_data.min())

normed_public_test_data = (public_test_data - public_test_data.min())/(public_test_data.max() - public_test_data.min())

normed_private_test_data = (private_test_data - private_test_data.min())/(private_test_data.max() - private_test_data.min())


In [8]:
normed_train_data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 250000 entries, 0 to 249999
Data columns (total 24 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   DER_mass_MMC                 250000 non-null  float64
 1   DER_mass_transverse_met_lep  250000 non-null  float64
 2   DER_mass_vis                 250000 non-null  float64
 3   DER_pt_h                     250000 non-null  float64
 4   DER_deltaeta_jet_jet         250000 non-null  float64
 5   DER_mass_jet_jet             250000 non-null  float64
 6   DER_prodeta_jet_jet          250000 non-null  float64
 7   DER_deltar_tau_lep           250000 non-null  float64
 8   DER_pt_tot                   250000 non-null  float64
 9   DER_sum_pt                   250000 non-null  float64
 10  DER_pt_ratio_lep_tau         250000 non-null  float64
 11  DER_lep_eta_centrality       250000 non-null  float64
 12  PRI_tau_pt                   250000 non-null  float64
 13 

In [9]:
train_input_data = []
for i in range(len(orig_train_data)):
   train_input_data.append([torch.tensor(normed_train_data.iloc[i], dtype=torch.float), torch.tensor(orig_train_data['Binary_Label'].iloc[i] , dtype=torch.float)])
valid_input_data = []
for i in range(len(orig_public_test_data)):
   valid_input_data.append([torch.tensor(normed_public_test_data.iloc[i], dtype=torch.float), torch.tensor(orig_public_test_data['Binary_Label'].iloc[i] , dtype=torch.float)])

In [10]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_input_data, batch_size=128, shuffle=True)
valid_dataloader = DataLoader(valid_input_data, batch_size=128, shuffle=True)

In [11]:
model = Feedforward(24, 600, 0.05)
criterion = torch.nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 0.01)

In [12]:
import time
import copy
from datetime import datetime

now = datetime.now()
dt_string = now.strftime("%d%m%Y%H%M%S")
logfile = open(dt_string + ".log",'w')

start_time = time.time()
epoch = 20
maxvalcount = 5
minval_loss = np.inf
valcount = maxvalcount
best_model = model
for epoch in range(epoch):
    print("EPOCH: ", epoch)
    model.train()
    for i,batch in enumerate(train_dataloader):
        inputs, output = batch
        optimizer.zero_grad()
        # Forward pass
        y_pred = model(inputs)
        # Compute Loss
        loss = criterion(y_pred.squeeze(), output)
        if i % 100 == 0:
            logstring = 'Batch {}: train loss: {}'.format(i, loss.item())
            print(logstring)
            logfile.write(logstring + '\n')
        # Backward pass
        loss.backward()
        optimizer.step()
    # compute validation loss
    model.eval()
    with torch.set_grad_enabled(False):
        val_loss = 0
        for i,batch in enumerate(valid_dataloader):
            inputs, output = batch
            y_pred = model(inputs)
            y_pred = model(inputs)
            loss = criterion(y_pred.squeeze(), output)
            val_loss += loss
            if i % 100 == 0:
                logstring = 'Validation Batch {}: loss: {}'.format(i, loss.item())
                print(logstring)
                logfile.write(logstring + '\n')
        avg_val_loss = val_loss / len(valid_dataloader)
        valstring = f"AVERAGE BATCH VAL LOSS = {avg_val_loss}"
        print(valstring)
        logfile.write(valstring + '\n')
    if avg_val_loss < minval_loss:
        minval_loss = avg_val_loss
        best_model = copy.deepcopy(model)
        valcount = maxvalcount
    else:
        valcount -= 1
    if valcount == 0:
        endmsg = f"Validation Loss failed to decrease in {maxvalcount} epochs, exiting with best model, val loss = {minval_loss}"
        print(endmsg)
        logfile.write(endmsg + '\n')
        break

end_time = time.time()
timestring = f"Time elapsed in training: {end_time - start_time} seconds"
print(timestring)
logfile.write(timestring + '\n')

EPOCH:  0
Batch 0: train loss: 0.6917551159858704
Batch 100: train loss: 0.6666089296340942
Batch 200: train loss: 0.6316730976104736
Batch 300: train loss: 0.6139228940010071
Batch 400: train loss: 0.6790646910667419
Batch 500: train loss: 0.6557291150093079
Batch 600: train loss: 0.6449435353279114
Batch 700: train loss: 0.6652800440788269
Batch 800: train loss: 0.6344177722930908
Batch 900: train loss: 0.6423394083976746
Batch 1000: train loss: 0.6257915496826172
Batch 1100: train loss: 0.6098276972770691
Batch 1200: train loss: 0.6416054964065552
Batch 1300: train loss: 0.5883806347846985
Batch 1400: train loss: 0.6087237596511841
Batch 1500: train loss: 0.6166571974754333
Batch 1600: train loss: 0.6526719331741333
Batch 1700: train loss: 0.6648691296577454
Batch 1800: train loss: 0.60236656665802
Batch 1900: train loss: 0.6082302331924438
Validation Batch 0: loss: 0.6097041368484497
Validation Batch 100: loss: 0.5880580544471741
Validation Batch 200: loss: 0.626838207244873
Valida

Batch 1300: train loss: 0.40102192759513855
Batch 1400: train loss: 0.46288397908210754
Batch 1500: train loss: 0.48143908381462097
Batch 1600: train loss: 0.4813075065612793
Batch 1700: train loss: 0.4656256139278412
Batch 1800: train loss: 0.44055482745170593
Batch 1900: train loss: 0.42344292998313904
Validation Batch 0: loss: 0.5646031498908997
Validation Batch 100: loss: 0.5968765616416931
Validation Batch 200: loss: 0.5530039668083191
Validation Batch 300: loss: 0.5807088613510132
Validation Batch 400: loss: 0.5985207557678223
Validation Batch 500: loss: 0.5514533519744873
Validation Batch 600: loss: 0.5873332619667053
Validation Batch 700: loss: 0.6319698691368103
AVERAGE BATCH VAL LOSS = 0.5834447741508484
EPOCH:  7
Batch 0: train loss: 0.542596161365509
Batch 100: train loss: 0.4183816909790039
Batch 200: train loss: 0.41279712319374084
Batch 300: train loss: 0.4794159233570099
Batch 400: train loss: 0.4548145532608032
Batch 500: train loss: 0.5760714411735535
Batch 600: train

Validation Batch 400: loss: 0.484864205121994
Validation Batch 500: loss: 0.5356382131576538
Validation Batch 600: loss: 0.47629883885383606
Validation Batch 700: loss: 0.5585200786590576
AVERAGE BATCH VAL LOSS = 0.5376240611076355
EPOCH:  13
Batch 0: train loss: 0.40627044439315796
Batch 100: train loss: 0.4003274440765381
Batch 200: train loss: 0.42251259088516235
Batch 300: train loss: 0.4075794816017151
Batch 400: train loss: 0.5216246843338013
Batch 500: train loss: 0.4910753667354584
Batch 600: train loss: 0.33936256170272827
Batch 700: train loss: 0.40404874086380005
Batch 800: train loss: 0.4739995300769806
Batch 900: train loss: 0.5096544623374939
Batch 1000: train loss: 0.4021013379096985
Batch 1100: train loss: 0.4315601587295532
Batch 1200: train loss: 0.37040552496910095
Batch 1300: train loss: 0.4129425585269928
Batch 1400: train loss: 0.4029676616191864
Batch 1500: train loss: 0.43003198504447937
Batch 1600: train loss: 0.3570355176925659
Batch 1700: train loss: 0.365477

Batch 600: train loss: 0.5407666563987732
Batch 700: train loss: 0.46103450655937195
Batch 800: train loss: 0.4292481243610382
Batch 900: train loss: 0.36585667729377747
Batch 1000: train loss: 0.42342036962509155
Batch 1100: train loss: 0.28076907992362976
Batch 1200: train loss: 0.48628538846969604
Batch 1300: train loss: 0.3587314784526825
Batch 1400: train loss: 0.36070677638053894
Batch 1500: train loss: 0.48541581630706787
Batch 1600: train loss: 0.4771183729171753
Batch 1700: train loss: 0.4355328381061554
Batch 1800: train loss: 0.44630977511405945
Batch 1900: train loss: 0.4355911910533905
Validation Batch 0: loss: 0.4910696744918823
Validation Batch 100: loss: 0.47192564606666565
Validation Batch 200: loss: 0.5179162621498108
Validation Batch 300: loss: 0.49711155891418457
Validation Batch 400: loss: 0.4362960755825043
Validation Batch 500: loss: 0.4333142936229706
Validation Batch 600: loss: 0.5174903273582458
Validation Batch 700: loss: 0.4869093894958496
AVERAGE BATCH VAL 

52

In [13]:
torch.save(best_model, '/Users/nkerstingadxnet.com/Documents/Higgs/output/mlp.600.nophi.pt')

In [14]:
best_model.eval()
y_pred = model(torch.tensor(normed_train_data.values, dtype=torch.float)).tolist()

In [67]:
import math
def AMS(s, b):
    """ Approximate Median Significance defined as:
        AMS = sqrt(
                2 { (s + b + b_r) log[1 + (s/(b+b_r))] - s}
              )        
    where b_r = 10, b = background, s = signal, log is natural logarithm """
    
    br = 10.0
    radicand = 2 *( (s+b+br) * math.log (1.0 + s/(b+br)) -s)
    if radicand < 0:
        print('radicand is negative. Exiting')
        exit()
    else:
        return math.sqrt(radicand)

In [68]:
# Training performance
y_gold = orig_train_data['Label']
y_weight = orig_train_data['KaggleWeight']
s = 0
b = 0
tp = 0
fp = 0
for i,y in enumerate(y_pred):
    if y[0] > 0.5:
        if y_gold.iloc[i] == 's':
            s += y_weight.iloc[i]
            tp += 1
        else:
            b += y_weight.iloc[i]
            fp += 1
print(f"TRAINING: (S,B) = ({s:.3f},{b:.3f}), AMS={AMS(s,b):.3f}, Unweighted Precision = {tp/(tp+fp):.3f}")

TRAINING: (S,B) = (592.949,70954.077), AMS=2.223, Unweighted Precision = 0.617


In [69]:
y_pred = best_model(torch.tensor(normed_public_test_data.values, dtype=torch.float)).tolist()

In [70]:
y_gold = orig_public_test_data['Label']
y_weight = orig_public_test_data['KaggleWeight']
s = 0
b = 0
tp = 0
fp = 0
for i,y in enumerate(y_pred):
    if y[0] > 0.5:
        if y_gold.iloc[i] == 's':
            s += y_weight.iloc[i]
            tp += 1
        else:
            b += y_weight.iloc[i]
            fp += 1
print(f"VALID: (S,B) = ({s:.3f},{b:.3f}), AMS={AMS(s,b):.3f}, Unweighted Precision = {tp/(tp+fp):.3f}")

VALID: (S,B) = (439.916,29919.952), AMS=2.537, Unweighted Precision = 0.722


In [71]:
y_pred = best_model(torch.tensor(normed_private_test_data.values, dtype=torch.float)).tolist()

In [72]:
y_gold = orig_private_test_data['Label']
y_weight = orig_private_test_data['KaggleWeight']
s = 0
b = 0
tp = 0
fp = 0
for i,y in enumerate(y_pred):
    if y[0] > 0.5:
        if y_gold.iloc[i] == 's':
            s += y_weight.iloc[i]
            tp += 1
        else:
            b += y_weight.iloc[i]
            fp += 1
print(f"TEST: (S,B) = ({s:.3f},{b:.3f}), AMS={AMS(s,b):.3f}, Unweighted Precision = {tp/(tp+fp):.3f}")

TEST: (S,B) = (352.093,21353.519), AMS=2.402, Unweighted Precision = 0.750


In [29]:
normed_public_test_data.head()

,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,DER_sum_pt,...,PRI_lep_pt,PRI_lep_eta,PRI_met,PRI_met_sumet,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_all_pt
250000,0.101990,0.087702,0.016287,0.002270,0.286389,0.091591,0.48232,0.119433,0.003998,0.005997,...,0.011742,0.399080,0.038546,0.047809,0.0,0.048355,0.499487,0.048145,0.499026,-0.0
250008,0.055259,0.073090,0.053375,0.000567,0.286389,0.091591,0.48232,0.389400,0.000998,0.012431,...,0.022059,0.726345,0.024009,0.049964,0.0,0.048355,0.499487,0.048145,0.499026,-0.0
250017,0.027302,0.084231,0.027383,0.044529,0.286389,0.091591,0.48232,0.118881,0.078412,0.021463,...,0.044543,0.038208,0.021982,0.107759,0.0,0.048355,0.499487,0.048145,0.499026,0.0
250027,0.088151,0.088579,0.065443,0.030233,0.286389,0.091591,0.48232,0.464299,0.053238,0.016104,...,0.025524,0.902581,0.032859,0.082216,0.0,0.048355,0.499487,0.048145,0.499026,-0.0
250032,0.129668,0.061777,0.087233,0.020138,0.286389,0.091591,0.48232,0.541038,0.035461,0.016073,...,0.016909,0.321464,0.017559,0.037446,0.0,0.048355,0.499487,0.048145,0.499026,0.0


In [30]:
orig_public_test_data.head()

,EventId,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,...,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_all_pt,Weight,Label,KaggleSet,KaggleWeight,Binary_Label
250000,350000,NaN,79.589,23.916,3.036,NaN,NaN,NaN,0.903,3.036,...,NaN,NaN,NaN,NaN,-0.0,1.380904,b,b,11.246677,0
250008,350008,70.958,66.329,60.950,0.758,NaN,NaN,NaN,2.370,0.758,...,NaN,NaN,NaN,NaN,-0.0,1.412085,b,b,11.500627,0
250017,350017,40.383,76.439,34.996,59.543,NaN,NaN,NaN,0.900,59.543,...,NaN,NaN,NaN,NaN,0.0,0.338196,b,b,2.754413,0
250027,350027,106.932,80.385,73.001,40.427,NaN,NaN,NaN,2.777,40.427,...,NaN,NaN,NaN,NaN,-0.0,0.005721,s,b,0.046987,1
250032,350032,152.338,56.063,94.760,26.928,NaN,NaN,NaN,3.194,26.928,...,NaN,NaN,NaN,NaN,0.0,0.005721,s,b,0.046987,1


In [63]:
from datetime import datetime

now = datetime.now()
dt_string = now.strftime("%d%m%Y%H%M%S")
print("date and time =", dt_string)	

date and time = 02112021102314
